[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.5_multi_region_kv_locality/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.5_multi_region_kv_locality/lab.ipynb)

# Lab 8.5: Multi-Region KV Cache Locality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.5_multi_region_kv_locality/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.ai/open?repo=harshuljain13/llm-inference-at-scale&path=content/09_operations/08.5_multi_region_kv_locality/lab.ipynb&branch=master)

This lab models the transfer-vs-recompute crossover point and visualizes multi-region routing decisions.

In [ ]:
# Install dependencies in subprocess to avoid kernel restart
import subprocess
import sys
# numpy and matplotlib are the only requirements for this simulation lab
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
import numpy as np  # numerical computation for cost modeling
import matplotlib.pyplot as plt  # visualization of crossover analysis

# --- Model Configuration (Llama 70B with GQA) ---
NUM_KV_HEADS = 8  # grouped-query attention uses 8 KV heads
HEAD_DIM = 128  # dimension per attention head
NUM_LAYERS = 80  # transformer layers in Llama 70B
DTYPE_BYTES = 2  # FP16 = 2 bytes per element

# Compute bytes of KV cache per token
# Formula: 2 (K and V) * num_kv_heads * head_dim * num_layers * dtype_bytes
KV_BYTES_PER_TOKEN = 2 * NUM_KV_HEADS * HEAD_DIM * NUM_LAYERS * DTYPE_BYTES
print(f"KV cache per token: {KV_BYTES_PER_TOKEN:,} bytes ({KV_BYTES_PER_TOKEN/1024:.1f} KB)")

In [ ]:
def compute_transfer_time_ms(num_tokens, bandwidth_gbps, protocol_efficiency=0.75):
    """Calculate KV cache transfer time in milliseconds.
    
    Transfer time grows linearly with context length.
    """
    # Total KV cache size in bytes
    total_bytes = num_tokens * KV_BYTES_PER_TOKEN
    # Convert bandwidth from Gbps to bytes/second with efficiency factor
    effective_bw_bytes_per_sec = bandwidth_gbps * 1e9 / 8 * protocol_efficiency
    # Time in milliseconds
    transfer_ms = (total_bytes / effective_bw_bytes_per_sec) * 1000
    return transfer_ms


def compute_prefill_time_ms(num_tokens, gpu_tflops=989, mfu=0.50):
    """Estimate prefill time on H100 in milliseconds.
    
    Prefill grows quadratically due to attention (O(n^2)).
    """
    # Attention FLOPs: 2 * layers * heads * seq^2 * head_dim
    # Using full attention heads (32) for compute, not KV heads
    NUM_ATTENTION_HEADS = 64  # Llama 70B has 64 query heads
    HIDDEN_DIM = 8192  # hidden dimension of Llama 70B
    attention_flops = 2 * NUM_LAYERS * NUM_ATTENTION_HEADS * (num_tokens ** 2) * HEAD_DIM
    # MLP FLOPs: 2 * layers * seq * 8 * hidden^2 (SwiGLU)
    mlp_flops = 2 * NUM_LAYERS * num_tokens * 8 * HIDDEN_DIM ** 2
    # Total compute needed
    total_flops = attention_flops + mlp_flops
    # Effective FLOPS with MFU (model FLOPS utilization)
    effective_flops = gpu_tflops * 1e12 * mfu
    # Convert to milliseconds
    prefill_ms = (total_flops / effective_flops) * 1000
    return prefill_ms

In [ ]:
# --- Crossover Analysis: Transfer vs Re-Prefill ---
# Sweep context lengths from 256 to 32K tokens
context_lengths = np.array([256, 512, 1024, 2048, 4096, 8192, 16384, 32768])
# These represent the range from short chats to long document contexts

# Compute times for two bandwidth scenarios
transfer_100gbps = [compute_transfer_time_ms(n, 100) for n in context_lengths]  # inter-region TCP
transfer_400gbps = [compute_transfer_time_ms(n, 400) for n in context_lengths]  # intra-DC RDMA
prefill_times = [compute_prefill_time_ms(n) for n in context_lengths]  # H100 prefill

# Plot the crossover
fig_c3, ax_c3 = plt.subplots(1, 1, figsize=(10, 6))
# Log scale makes the quadratic vs linear relationship visible
ax_c3.semilogy(context_lengths, prefill_times, 'r-o', linewidth=2, label='Re-prefill (H100, O(n^2))')
ax_c3.semilogy(context_lengths, transfer_100gbps, 'b-s', linewidth=2, label='Transfer @ 100 Gbps (inter-region)')
ax_c3.semilogy(context_lengths, transfer_400gbps, 'g-^', linewidth=2, label='Transfer @ 400 Gbps (intra-DC RDMA)')

# Mark the crossover zones with vertical lines
ax_c3.axvline(x=4096, color='blue', linestyle='--', alpha=0.5, label='Crossover @ 100 Gbps (~4K tokens)')
ax_c3.axvline(x=1024, color='green', linestyle='--', alpha=0.5, label='Crossover @ 400 Gbps (~1K tokens)')

ax_c3.set_xlabel('Context Length (tokens)', fontsize=12)
ax_c3.set_ylabel('Time (ms, log scale)', fontsize=12)
ax_c3.set_title('KV Cache Transfer vs Re-Prefill: The Crossover Point', fontsize=14)
ax_c3.legend(loc='upper left', fontsize=10)
ax_c3.grid(True, alpha=0.3)
ax_c3.set_xticks(context_lengths)
ax_c3.set_xticklabels([f'{n//1024}K' if n >= 1024 else str(n) for n in context_lengths], fontsize=9)
plt.tight_layout()
plt.show()

# Print the exact crossover data
print("\nCrossover Table (Llama 70B, GQA 8 KV heads):")
print(f"{'Context':<10} {'KV Size':<12} {'Transfer@100G':<15} {'Transfer@400G':<15} {'Prefill':<12} {'Winner@100G':<12}")
for i, n in enumerate(context_lengths):
    # Calculate KV size for display
    kv_mb = n * KV_BYTES_PER_TOKEN / 1e6
    # Determine which strategy wins at 100 Gbps
    winner = 'Transfer' if transfer_100gbps[i] < prefill_times[i] else 'Re-prefill'
    print(f"{n:<10} {kv_mb:<12.1f}MB {transfer_100gbps[i]:<15.1f}ms {transfer_400gbps[i]:<15.1f}ms {prefill_times[i]:<12.1f}ms {winner:<12}")

In [ ]:
# --- Multi-Region Routing Simulation ---
# Simulate routing decisions for 1000 requests with varying context lengths

np.random.seed(42)  # reproducible results

# Generate realistic context length distribution (log-normal)
# Most conversations are short, some are very long
num_requests = 1000
context_dist = np.random.lognormal(mean=7.5, sigma=1.2, size=num_requests).astype(int)
# Clip to realistic range (128 to 32K tokens)
context_dist = np.clip(context_dist, 128, 32768)
# Realistic range: shortest useful prompt to max context window

# Simulate routing decisions with 1.3x hysteresis factor
HYSTERESIS = 1.3  # transfer must be 30% faster to justify complexity
# 30% margin accounts for network reliability risk
BANDWIDTH_GBPS = 100  # inter-region bandwidth

decisions = []  # store routing decision per request
for ctx_len in context_dist:
    # Compute both options
    t_transfer = compute_transfer_time_ms(ctx_len, BANDWIDTH_GBPS)
    t_prefill = compute_prefill_time_ms(ctx_len)
    # Apply hysteresis: transfer must beat prefill by 30%
    if t_transfer * HYSTERESIS < t_prefill:
        decisions.append('transfer')
    else:
        decisions.append('re-prefill')

# Compute statistics
transfer_count = decisions.count('transfer')  # requests routed via KV transfer
prefill_count = decisions.count('re-prefill')  # requests using fresh prefill
transfer_pct = transfer_count / num_requests * 100

# Visualize the decision distribution
fig_c4, (ax1_c4, ax2_c4) = plt.subplots(1, 2, figsize=(12, 5))

# Left: histogram of context lengths colored by decision
transfer_contexts = [ctx_dist for ctx_dist, d in zip(context_dist, decisions) if d == 'transfer']
prefill_contexts = [ctx_dist for ctx_dist, d in zip(context_dist, decisions) if d == 're-prefill']
ax1_c4.hist(prefill_contexts, bins=30, alpha=0.7, color='#ef4444', label=f'Re-prefill ({prefill_count})')
ax1_c4.hist(transfer_contexts, bins=30, alpha=0.7, color='#3b82f6', label=f'Transfer ({transfer_count})')
ax1_c4.set_xlabel('Context Length (tokens)', fontsize=11)
ax1_c4.set_ylabel('Number of Requests', fontsize=11)
ax1_c4.set_title('Routing Decisions by Context Length', fontsize=13)
ax1_c4.legend(fontsize=10)
ax1_c4.grid(True, alpha=0.3)

# Right: pie chart of overall decision split
ax2_c4.pie([prefill_count, transfer_count],
        labels=[f'Re-prefill\n({prefill_pct:.0f}%)', f'Transfer\n({transfer_pct:.0f}%)'],
        colors=['#fecaca', '#bfdbfe'],
        explode=[0, 0.05],
        shadow=True,
        textprops={'fontsize': 12})
ax2_c4.set_title(f'Overall Routing Split\n(1.3x hysteresis, 100 Gbps)', fontsize=13)

plt.tight_layout()
plt.show()

print(f"\nResults: {transfer_pct:.1f}% of requests benefit from KV transfer (context > ~4K tokens)")
print(f"Median context in transfer group: {np.median(transfer_contexts):.0f} tokens")

In [ ]:
# --- Prefix Pool Savings Calculator ---
# Model the ROI of pre-computing and replicating common prefixes globally

# Parameters (enterprise chatbot scenario)
SYSTEM_PROMPT_TOKENS = 1500  # fixed system prompt length
FEW_SHOT_TOKENS = 800  # fixed few-shot examples
PREFIX_TOKENS = SYSTEM_PROMPT_TOKENS + FEW_SHOT_TOKENS  # total cacheable prefix
# This is the portion we can pre-compute and cache globally
DAILY_REQUESTS = 10_000_000  # 10M requests per day
GPU_COST_PER_HOUR = 30.0  # H100 on-demand price in dollars
NUM_REGIONS = 5  # global deployment across 5 regions

# Compute prefill time savings from prefix caching
# Without prefix cache: must prefill all PREFIX_TOKENS for every request
prefill_time_per_request_ms = compute_prefill_time_ms(PREFIX_TOKENS)
# Convert to GPU-seconds saved per day
gpu_seconds_saved_per_day = DAILY_REQUESTS * (prefill_time_per_request_ms / 1000)
# Convert to dollar savings
gpu_cost_per_second = GPU_COST_PER_HOUR / 3600
daily_savings = gpu_seconds_saved_per_day * gpu_cost_per_second
annual_savings = daily_savings * 365

# Compute prefix pool memory cost
# Each prefix KV cache size
prefix_kv_bytes = PREFIX_TOKENS * KV_BYTES_PER_TOKEN
prefix_kv_gb = prefix_kv_bytes / 1e9
# Assume 50 popular prefixes replicated across all regions
NUM_PREFIXES = 50
total_prefix_memory_gb = NUM_PREFIXES * prefix_kv_gb * NUM_REGIONS
# Cost: memory in GPU HBM (fraction of a GPU)
gpus_for_prefix = total_prefix_memory_gb / 80  # H100 has 80 GB HBM
monthly_prefix_cost = gpus_for_prefix * GPU_COST_PER_HOUR * 720  # hours in a month

# Print the ROI analysis
print("=== Prefix Pool ROI Analysis ===")
print(f"\nPrefix length: {PREFIX_TOKENS:,} tokens ({prefix_kv_gb*1000:.0f} MB KV cache)")
print(f"Daily requests: {DAILY_REQUESTS:,}")
print(f"Prefill saved per request: {prefill_time_per_request_ms:.1f} ms")
print(f"\n--- Savings ---")
print(f"GPU-seconds saved/day: {gpu_seconds_saved_per_day:,.0f} ({gpu_seconds_saved_per_day/86400:.1f} GPU-days)")
print(f"Daily savings: ${daily_savings:,.0f}")
print(f"Annual savings: ${annual_savings:,.0f}")
print(f"\n--- Costs ---")
print(f"Prefix memory needed: {total_prefix_memory_gb:.1f} GB across {NUM_REGIONS} regions")
print(f"GPUs dedicated to prefix cache: {gpus_for_prefix:.1f}")
print(f"Monthly prefix storage cost: ${monthly_prefix_cost:,.0f}")
print(f"\n--- ROI ---")
net_monthly_savings = (daily_savings * 30) - monthly_prefix_cost
# Positive value means prefix caching is profitable
print(f"Net monthly savings: ${net_monthly_savings:,.0f}")
print(f"ROI: {net_monthly_savings / monthly_prefix_cost * 100:.0f}% per month" if monthly_prefix_cost > 0 else "Infinite ROI")